In [1]:
# Install Ultralytics YOLO (PyTorch is already available in Colab)
!pip install ultralytics --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.5 MB/s eta 0:00:00


In [2]:
import torch

print("PyTorch version:", torch.__version__)
print("Is CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))


PyTorch version: 2.10.0+cu128
Is CUDA available: True
GPU name: Tesla T4


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os

# Change this path according to where you placed the folder in Drive
BASE_DIR = "/content/drive/MyDrive/Pothole.fv-raw.yolov8"

train_dir = os.path.join(BASE_DIR, "train")
val_dir   = os.path.join(BASE_DIR, "valid")
test_dir  = os.path.join(BASE_DIR, "test")
data_yaml = os.path.join(BASE_DIR, "data.yaml")

print("BASE_DIR:", BASE_DIR)
print("Train dir:", train_dir)
print("Val dir:", val_dir)
print("Test dir:", test_dir)

print("\nFiles in BASE_DIR:")
print(os.listdir(BASE_DIR))


BASE_DIR: /content/drive/MyDrive/Pothole.fv-raw.yolov8
Train dir: /content/drive/MyDrive/Pothole.fv-raw.yolov8/train
Val dir: /content/drive/MyDrive/Pothole.fv-raw.yolov8/valid
Test dir: /content/drive/MyDrive/Pothole.fv-raw.yolov8/test

Files in BASE_DIR:
['sample_video.mp4', 'train', 'valid', 'data.yaml']


In [5]:
%%writefile /content/drive/MyDrive/Pothole.fv-raw.yolov8/data.yaml
path: /content/drive/MyDrive/Pothole.fv-raw.yolov8

train: train/images
val: valid/images

nc: 1
names:
  0: pothole


Overwriting /content/drive/MyDrive/Pothole.fv-raw.yolov8/data.yaml


In [6]:
!cat /content/drive/MyDrive/Pothole.fv-raw.yolov8/data.yaml


path: /content/drive/MyDrive/Pothole.fv-raw.yolov8

train: train/images
val: valid/images

nc: 1
names:
  0: pothole


In [7]:
!cp -r /content/drive/MyDrive/Pothole.fv-raw.yolov8 /content/pothole_dataset


In [8]:
from ultralytics import YOLO

model = YOLO("yolov8m.pt")

model.train(
    data="/content/pothole_dataset/data.yaml",  # ✅ local path
    epochs=80,
    patience=20,
    imgsz=640,          # 🔥 faster, still accurate
    batch=16,           # better GPU utilization
    optimizer="AdamW",
    lr0=0.001,
    weight_decay=0.0005,
    mosaic=0.3,
    mixup=0.05,
    hsv_h=0.01,
    hsv_s=0.4,
    hsv_v=0.3,
    degrees=3,
    translate=0.05,
    scale=0.4,
    fliplr=0.5,
    device=0,
    project="/content/YOLO_RESULTS",   # save locally
    name="pothole_yolov8m",
    plots=True,
    workers=2
)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/pothole_dataset/data.yaml, degrees=3, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.01, hsv_s=0.4, hsv_v=0.3, imgsz=640, int

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7aa019c205c0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 

In [ ]:
# Use the test images folder from your dataset
test_images_dir = os.path.join(test_dir, "images")

# Make predictions and save annotated images
pred_results = model.predict(
    source=test_images_dir,
    save=True  # saves annotated images in runs/detect/predict
)

# Show first few prediction files created
pred_dir = "runs/detect"
print("Prediction runs:", os.listdir(pred_dir))



image 1/67 /content/drive/MyDrive/Pothole.v1-raw.yolov8/test/images/img-105_jpg.rf.3fe9dff3d1631e79ecb480ff403bcb86.jpg: 960x960 2 potholes, 29.5ms
image 2/67 /content/drive/MyDrive/Pothole.v1-raw.yolov8/test/images/img-107_jpg.rf.2e40485785f6e5e2efec404301b235c2.jpg: 960x960 2 potholes, 29.5ms
image 3/67 /content/drive/MyDrive/Pothole.v1-raw.yolov8/test/images/img-146_jpg.rf.61be25b3053a51f622a244980545df2b.jpg: 960x960 2 potholes, 29.5ms
image 4/67 /content/drive/MyDrive/Pothole.v1-raw.yolov8/test/images/img-161_jpg.rf.211541e7178a4a93ec0680f26b905427.jpg: 960x960 2 potholes, 29.5ms
image 5/67 /content/drive/MyDrive/Pothole.v1-raw.yolov8/test/images/img-168_jpg.rf.af3590e07b06b43e91fa53990ff94af3.jpg: 960x960 3 potholes, 29.4ms
image 6/67 /content/drive/MyDrive/Pothole.v1-raw.yolov8/test/images/img-179_jpg.rf.8632eb0d9b75fefe144829e67b75015a.jpg: 960x960 5 potholes, 29.5ms
image 7/67 /content/drive/MyDrive/Pothole.v1-raw.yolov8/test/images/img-195_jpg.rf.f77a8f4d432a9a89235168ff8e09

In [ ]:
import pandas as pd

df = pd.read_csv("/content/runs/detect/train/results.csv")
df.tail()


,epoch,time,train/box_loss,train/cls_loss,train/dfl_loss,metrics/precision(B),metrics/recall(B),metrics/mAP50(B),metrics/mAP50-95(B),val/box_loss,val/cls_loss,val/dfl_loss,lr/pg0,lr/pg1,lr/pg2
195,196,4954.29,1.20456,0.91263,1.39787,0.74671,0.65152,0.74063,0.38598,1.55762,1.23695,1.67173,0.000347,0.000347,0.000347
196,197,4977.50,1.21015,0.90730,1.41166,0.76862,0.62727,0.73380,0.37936,1.55883,1.22192,1.67500,0.000298,0.000298,0.000298
197,198,5001.00,1.18629,0.87831,1.40087,0.80031,0.61212,0.73704,0.37869,1.56545,1.21658,1.68883,0.000249,0.000249,0.000249
198,199,5023.31,1.18943,0.91363,1.40600,0.80189,0.63030,0.74452,0.38542,1.55319,1.20777,1.67794,0.000199,0.000199,0.000199
199,200,5046.36,1.17720,0.87039,1.39109,0.81667,0.62121,0.74207,0.38547,1.57013,1.20883,1.68740,0.000150,0.000150,0.000150
